# Notebook 07 — Network Epidemic Dynamics

**Module:** 15 — Simulation and Agent-Based Modeling  
**Tier:** 2 — Working competence  
**Notebook:** 7 of 12  
**Time estimate:** 75 minutes

> The SIR ODE assumes everyone contacts everyone else equally. Real populations
> have heterogeneous contact networks: some individuals are highly connected hubs,
> others are nearly isolated. This changes epidemic dynamics fundamentally — and
> shatters the herd immunity threshold on scale-free networks.

---
## Step 1 — Motivation

Pastor-Satorras & Vespignani (2001) showed that on a scale-free network (where
the degree distribution follows a power law), the epidemic threshold β_c → 0:
an epidemic can always spread, no matter how weak the transmission, because
high-degree hubs act as super-spreaders. This result holds in the thermodynamic
limit N→∞. It changed how epidemiologists think about targeted vaccination
(vaccinating hubs is much more efficient than random vaccination).

---
## Step 2 — Intuition

**Network SIR:** same S, I, R compartments as the ODE model, but:
- Each node is an individual
- Transmission only occurs along edges (contacts)
- At each step, each I node infects each S neighbour with probability β
- Each I node recovers with probability γ

**Erdős-Rényi (ER) graph:** random network where each pair of N nodes is connected
independently with probability p. Degree distribution is Poisson(pN) — homogeneous.

**Barabási-Albert (BA) graph:** grown by preferential attachment — new nodes
connect to existing nodes proportional to their degree. Degree distribution is
power law P(k) ~ k^{-3}. A few nodes have very high degree (hubs), most have
few connections — like the internet, social networks, protein interaction networks.

**Epidemic threshold (mean-field):**
- ER: $\beta_c = \gamma / \langle k \rangle$ (same as ODE SIR with mean degree)
- BA: $\beta_c \propto \langle k \rangle / \langle k^2 \rangle \to 0$ as N→∞
  (because variance of degree diverges — hubs guarantee spread)

---
## Step 3 — Biological Background

**Contact network examples in biology:**
- Social networks: sexual contact networks for STI spread are scale-free
  (Liljeros 2001) — small number of highly active individuals drive most transmission.
- Protein-protein interaction networks are scale-free — most proteins interact
  with few others, but essential hub proteins interact with hundreds.
- Airline networks: highly connected hub airports are responsible for the rapid
  global spread of respiratory infections (SARS 2003, COVID-19 2020).
- Epidemiology implication: targeted vaccination of hubs can control an epidemic
  with much less coverage than random vaccination (Ring et al. 2016).

**Household structure:** in reality, contact networks are hierarchical —
household contacts are daily, workplace contacts weekly, social contacts monthly.
The effective R₀ depends on the contact rate at each level.

**Superspreaders (observed):** in SARS 2003, ~20% of cases caused 80% of
transmission events (20-80 rule / negative binomial overdispersion k~0.16).
This is a property of the contact network combined with heterogeneous
infectivity.

---
## Step 4 — Mathematical Explanation

**Discrete-time network SIR (one simulation step):**
$$P(S_i \to I_i) = 1 - (1-\beta)^{k_i^{(I)}}$$
where $k_i^{(I)}$ = number of infectious neighbours of node $i$.
Equivalently: at least one infectious neighbour succeeds in transmitting.

**Final size relation (on ER network):** same transcendental equation as ODE SIR:
$r_\infty = 1 - e^{-R_0 r_\infty}$, where $R_0 = \beta \langle k \rangle / \gamma$.

**Degree-heterogeneous mean-field (Pastor-Satorras & Vespignani):**
Let $\rho_k(t)$ = fraction of degree-k nodes that are infected at time t.
$$\dot{\rho}_k = -\gamma \rho_k + \beta k (1 - \rho_k) \Theta(t)$$
where $\Theta(t) = \sum_k P(k) k \rho_k / \langle k \rangle$ is the probability
that a random neighbour is infected. Epidemic threshold:
$$\beta_c = \frac{\gamma \langle k \rangle}{\langle k^2 \rangle}$$
For a power-law degree distribution with exponent 2 < γ ≤ 3: $\langle k^2 \rangle \to \infty$,
so $\beta_c \to 0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import networkx as nx
    NX_AVAILABLE = True
except ImportError:
    NX_AVAILABLE = False
    print('NetworkX not available — using synthetic adjacency lists')

rng = np.random.default_rng(42)

# ---- Graph generators (with fallback if networkx missing) ----
if NX_AVAILABLE:
    def make_er(N, mean_degree, seed):
        p = mean_degree / N
        return nx.erdos_renyi_graph(N, p, seed=seed)

    def make_ba(N, mean_degree, seed):
        m = mean_degree // 2
        return nx.barabasi_albert_graph(N, m, seed=seed)

    def graph_adjacency(G):
        """Return dict: node -> list of neighbours."""
        return {n: list(G.neighbors(n)) for n in G.nodes()}
else:
    def _random_er(N, mean_degree, seed):
        rng_g = np.random.default_rng(seed)
        adj = {i: [] for i in range(N)}
        p = mean_degree / N
        for i in range(N):
            for j in range(i+1, N):
                if rng_g.random() < p:
                    adj[i].append(j); adj[j].append(i)
        return adj

    def make_er(N, mean_degree, seed): return None  # placeholder
    def make_ba(N, mean_degree, seed): return None
    def graph_adjacency(G):
        return {}

# ---- Network SIR simulation from scratch ----
def network_sir(adj, N, beta, gamma, seed, i0_frac=0.01):
    """
    Discrete-time network SIR.
    adj: dict {node: [neighbours]}
    beta: per-contact transmission probability per step
    gamma: recovery probability per step
    i0_frac: initial fraction infected
    Returns: S(t), I(t), R(t) arrays
    """
    rng_sim = np.random.default_rng(seed)
    state = np.zeros(N, dtype=int)  # 0=S, 1=I, 2=R
    i0 = max(1, int(i0_frac * N))
    init_infected = rng_sim.choice(N, i0, replace=False)
    state[init_infected] = 1

    S_t, I_t, R_t = [], [], []
    for _ in range(500):
        S_t.append((state == 0).sum())
        I_t.append((state == 1).sum())
        R_t.append((state == 2).sum())
        if I_t[-1] == 0:
            break
        new_state = state.copy()
        # Recovery
        recovering = (state == 1) & (rng_sim.random(N) < gamma)
        new_state[recovering] = 2
        # Transmission: for each S node, count infected neighbours
        for node in range(N):
            if state[node] == 0:  # susceptible
                n_infected_neigh = sum(1 for nb in adj.get(node, []) if state[nb] == 1)
                if n_infected_neigh > 0:
                    p_infect = 1 - (1 - beta)**n_infected_neigh
                    if rng_sim.random() < p_infect:
                        new_state[node] = 1
        state = new_state
    return np.array(S_t)/N, np.array(I_t)/N, np.array(R_t)/N

# Build graphs
N = 500; MEAN_DEG = 6; BETA = 0.05; GAMMA = 0.1

if NX_AVAILABLE:
    G_er = make_er(N, MEAN_DEG, seed=42)
    G_ba = make_ba(N, MEAN_DEG, seed=42)
    adj_er = graph_adjacency(G_er)
    adj_ba = graph_adjacency(G_ba)
    deg_er = [d for _, d in G_er.degree()]
    deg_ba = [d for _, d in G_ba.degree()]
    print(f'ER graph: mean degree={np.mean(deg_er):.2f}, max={max(deg_er)}')
    print(f'BA graph: mean degree={np.mean(deg_ba):.2f}, max={max(deg_ba)}')
else:
    adj_er = _random_er(N, MEAN_DEG, seed=42)
    adj_ba = adj_er.copy()  # fallback: same graph for both
    deg_er = [len(v) for v in adj_er.values()]
    deg_ba = deg_er
    print('NetworkX not available: using ER for both graph types')

print('Running ER network SIR...')
S_er, I_er, R_er = network_sir(adj_er, N, BETA, GAMMA, seed=1)
print('Running BA network SIR...')
S_ba, I_ba, R_ba = network_sir(adj_ba, N, BETA, GAMMA, seed=1)
print(f'ER final infected fraction: {R_er[-1]:.3f}')
print(f'BA final infected fraction: {R_ba[-1]:.3f}')

In [ ]:
# ---- Sweep beta: final epidemic size vs. beta ----
beta_vals = np.arange(0.01, 0.15, 0.01)
final_er, final_ba = [], []
for bv in beta_vals:
    _, _, r_er = network_sir(adj_er, N, bv, GAMMA, seed=7)
    _, _, r_ba = network_sir(adj_ba, N, bv, GAMMA, seed=7)
    final_er.append(r_er[-1])
    final_ba.append(r_ba[-1])

# ODE SIR reference
def sir_ode_final(beta, gamma=0.1, N_pop=1000):
    """Numerically find the final size of SIR ODE: solve r = 1 - exp(-R0*r)."""
    R0 = beta * MEAN_DEG / gamma  # mean-field ODE with mean degree
    if R0 <= 1:
        return 0.0
    from scipy.optimize import brentq
    return brentq(lambda r: r - (1 - np.exp(-R0*r)), 1e-9, 1-1e-9)

final_ode = [sir_ode_final(bv) for bv in beta_vals]
print('Beta sweep complete.')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel A: Degree distributions
ax = axes[0, 0]
ax.hist(deg_er, bins=range(0, max(deg_er)+2), alpha=0.7, density=True,
          color='steelblue', label='Erdős-Rényi')
ax.hist(deg_ba, bins=range(0, min(max(deg_ba)+2, 60)), alpha=0.7, density=True,
          color='tomato', label='Barabási-Albert')
ax.set_xlabel('Degree k'); ax.set_ylabel('P(k)')
ax.set_title('A. Degree distributions\n(ER: Poisson, BA: power-law)')
ax.legend(fontsize=9)

# Panel B: Degree distribution log-log (for BA)
if max(deg_ba) > 10:
    ax = axes[0, 1]
    from collections import Counter
    deg_count_ba = Counter(deg_ba)
    k_vals = sorted(deg_count_ba.keys())
    p_vals_ba = [deg_count_ba[k]/N for k in k_vals]
    ax.loglog(k_vals, p_vals_ba, 'o', color='tomato', ms=5, label='BA degree dist.')
    # Power law reference k^{-3}
    k_range = np.linspace(1, max(k_vals), 100)
    ref = p_vals_ba[2] * (k_vals[2]/k_range)**3
    ax.loglog(k_range, ref, 'k--', lw=1.5, label='k⁻³ (reference)')
    ax.set_xlabel('Degree k'); ax.set_ylabel('P(k)')
    ax.set_title('B. BA degree distribution (log-log)\nconfirms power-law tail')
    ax.legend(fontsize=8)
else:
    axes[0,1].text(0.5, 0.5, 'NetworkX required\nfor log-log plot', ha='center', va='center',
                     transform=axes[0,1].transAxes)
    axes[0,1].set_title('B. BA degree log-log (needs NetworkX)')

# Panel C: Epidemic curves
ax = axes[0, 2]
t_er = range(len(I_er)); t_ba = range(len(I_ba))
ax.plot(t_er, I_er, 'steelblue', lw=2, label=f'ER (peak={max(I_er):.3f})')
ax.plot(t_ba, I_ba, 'tomato', lw=2, label=f'BA (peak={max(I_ba):.3f})')
ax.set_xlabel('Time step'); ax.set_ylabel('Fraction infectious')
ax.set_title(f'C. Epidemic curves on ER vs. BA\n(β={BETA}, γ={GAMMA}, N={N})')
ax.legend(fontsize=9)

# Panel D: Final epidemic size vs beta
ax = axes[1, 0]
ax.plot(beta_vals, final_er,  'steelblue', lw=2, marker='o', ms=5, label='ER network')
ax.plot(beta_vals, final_ba,  'tomato', lw=2, marker='s', ms=5, label='BA network')
ax.plot(beta_vals, final_ode, 'k--', lw=1.5, label='ODE SIR (mean-field)')
beta_c_er = GAMMA / MEAN_DEG
ax.axvline(beta_c_er, color='steelblue', ls=':', lw=1,
             label=f'ER threshold β_c={beta_c_er:.2f}')
ax.set_xlabel('Transmission probability β')
ax.set_ylabel('Final epidemic size')
ax.set_title('D. Epidemic threshold: ER vs. BA')
ax.legend(fontsize=7)

# Panel E: R and S at end
ax = axes[1, 1]
ax.plot(range(len(R_er)), R_er, 'steelblue', lw=2, label='ER recovered')
ax.plot(range(len(R_ba)), R_ba, 'tomato', lw=2, label='BA recovered')
ax.plot(range(len(S_er)), S_er, 'steelblue', lw=1.5, ls='--', label='ER susceptible')
ax.plot(range(len(S_ba)), S_ba, 'tomato', lw=1.5, ls='--', label='BA susceptible')
ax.set_xlabel('Time step'); ax.set_ylabel('Fraction')
ax.set_title('E. S(t) and R(t) on ER vs. BA')
ax.legend(fontsize=8)

# Panel F: Summary table
ax = axes[1, 2]
ax.axis('off')
rows = [
    ['Property', 'Erdős-Rényi', 'Barabási-Albert'],
    ['Degree dist.', 'Poisson(⟨k⟩)', 'Power-law ~k⁻³'],
    ['Max degree', f'{max(deg_er)}', f'{max(deg_ba)}'],
    ['Epidemic threshold', 'γ/⟨k⟩ > 0', '→ 0 as N→∞'],
    ['Targeted vaccine', 'Moderate gain', 'Huge gain (hub removal)'],
    ['Real world example', 'Random contacts', 'Sexual/social networks'],
]
table = ax.table(cellText=rows[1:], colLabels=rows[0],
                   loc='center', cellLoc='center')
table.auto_set_font_size(False); table.set_fontsize(9)
table.scale(1, 1.8)
ax.set_title('F. ER vs. BA network comparison')

plt.suptitle('Module 15 NB07: Network Epidemic Dynamics', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('network_epidemic_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement targeted vaccination: remove the top-k highest-degree nodes from
   the BA graph before running SIR. Compare the epidemic curve and final size
   to removing k random nodes. How many hubs need to be vaccinated to
   prevent an epidemic at β=0.05?
2. Run 10 stochastic replicate epidemics on the same BA graph with β=0.02
   (near the threshold). Plot all curves. What fraction of replicates produce
   a major epidemic (>10% infected)? What fraction go extinct early?
3. Implement the SEIR model on the BA network (add Exposed state with mean
   incubation period 1/σ). Show how the epidemic curve changes compared to SIR.

---
## Step 10 — Quiz

1. Why does the epidemic threshold vanish on scale-free networks in the
   thermodynamic limit? Show the mathematical argument using ⟨k⟩ and ⟨k²⟩.
2. Explain the preferential attachment mechanism in the Barabási-Albert model.
   What biological network-forming process does it resemble?
3. In the network SIR, how is the per-contact transmission probability β
   related to the ODE rate β? Are they the same parameter?
4. Why is targeted vaccination of hubs more efficient than random vaccination
   on a scale-free network? Quantify the difference.

---
## Step 12 — Reflection

> *[In one paragraph: explain why the finding that β_c → 0 on scale-free
> networks was so impactful for public health. What practical intervention
> strategies does it suggest? What does it imply for the feasibility of
> eradication of STIs like HIV through random vaccination campaigns?]*

---
*Next: `08_spatial_abm_movement.ipynb`*